In [1]:
import earthkit.data as ekd
import earthkit.plots as ekp
from earthkit.plots.resample import Bilinear, Subsample, NearestNeighbour, Regrid
from pathlib import Path
from earthkit.geo import regrid
import numpy as np

THUMBNAILS_DIR = Path("../../../_static/thumbnails/")

data = ekd.from_source("sample", "era5-2t-msl-1985122512.grib").to_fieldlist()
pressure = data.sel({"parameter.variable": "msl"})
temperature = data.sel({"parameter.variable": "2t"})

earthkit-data cache: filesystem usage (99%) exceeds limit (95%) set by 'maximum-cache-disk-usage' config option
earthkit-data cache: trying to free 18.4 GiB
earthkit-data cache: could not free 18.4 GiB


In [63]:
chart = ekp.Map(domain=["France", "Spain"], figsize=(2, 2))

chart.contourf(temperature, units="celsius", style="auto")
chart.contour(pressure, units="hPa", style="auto", linewidths=[0.4, 0.8])

chart.coastlines()
cbar = chart.legend(label="")
chart.title(fontsize=8, color="grey")

for t in cbar[0].ax.get_xticklabels():
    t.set_fontsize(8)
    t.set_color("grey")

chart.save(THUMBNAILS_DIR/"01-introduction.png", transparent=True)

In [67]:
import cartopy.crs as ccrs

chart = ekp.Map(crs=ccrs.NearsidePerspective(central_latitude=52, central_longitude=10), figsize=(2, 2))

chart.ocean(color="white")
chart.land(transform_first=False)
chart.coastlines(color="lightgrey")
chart.borders(linewidth=0.2, color="lightgrey")

chart.gridlines(draw_labels=False, ylim=70, xstep=10, ystep=10)
chart.gridlines(draw_labels=False, ylim=80, xstep=30, ystep=10)

chart.save(THUMBNAILS_DIR/"06-domains.png", transparent=True)

In [31]:
joachim = ekd.from_source(
    "url",
    "https://get.ecmwf.int/repository/test-data/metview/gallery/fc_msl_wg_joachim.grib",
).to_fieldlist()
import matplotlib.pyplot as plt

figure = ekp.Figure(domain=[-5, 23, 40, 58], figsize=(2, 1.6), rows=3, columns=4)

gust_style = ekp.styles.Style(
    colors=["#85AAEE", "#208EFC", "#6CA632", "#FFB000", "#FF0000", "#7A11B1"],
    levels=[12, 15, 20, 25, 30, 35, 50],
    units="m s-1",
)

# Start at the top-right cell, leaving a gap for the colour bar
figure.add_map(0, 3)
for i in range(8):
    figure.add_map(1 + i // 4, i % 4)

figure.contourf(joachim.sel({"parameter.variable": "10fg6"}), style=gust_style)
# figure.contour(joachim.sel({"parameter.variable": "msl"}), units="hPa", style="auto")

figure.land()
figure.coastlines(linewidth=0.2)
figure.borders(linewidth=0.1)

# Place the colourbar on a custom matplotlib axes
ax = plt.axes((0.05, 0.8, 0.65, 0.025))
cbar = figure.legend(ax=ax, label="")

# figure.subplot_titles("{time.valid_datetime:%Y-%m-%d %H}", fontsize=4, color="grey")
figure.title(
    "ECMWF HRES Run: {base_time:%Y-%m-%d %H} UTC\nMaximum 10m wind gust",
    fontsize=6, color="grey", horizontalalignment="left", x=0, y=0.96,
)

for t in cbar[0].ax.get_xticklabels():
    t.set_fontsize(6)
    t.set_color("grey")

figure.save(THUMBNAILS_DIR/"02-api-layers.png", transparent=True)

In [43]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from matplotlib.patches import Patch
import numpy as np

fig = plt.figure(figsize=(2, 2))

# Colorbar 1 - Spectral_r with extend='both'
ax1 = fig.add_axes([0.1, 0.8, 0.8, 0.08])
cmap1 = plt.cm.Spectral_r
bounds1 = np.arange(-20, 41, 4)
norm1 = mcolors.BoundaryNorm(bounds1, cmap1.N)
cb1 = mcolorbar.ColorbarBase(ax1, cmap=cmap1, norm=norm1,
                              orientation='horizontal', extend='max',
                              ticks=bounds1[::2])
cb1.ax.tick_params(labelsize=8, length=0, colors='grey')
cb1.outline.set_edgecolor('grey')
cb1.outline.set_linewidth(0.3)

# Colorbar 1 - Spectral_r with extend='both'
ax1 = fig.add_axes([0.1, 0.575, 0.8, 0.08])
cmap1 = plt.cm.RdBu_r
bounds1 = np.arange(-20, 21, 4)
norm1 = mcolors.BoundaryNorm(bounds1, cmap1.N)
cb1 = mcolorbar.ColorbarBase(ax1, cmap=cmap1, norm=norm1,
                              orientation='horizontal', extend='both',
                              ticks=bounds1[::2])
cb1.ax.tick_params(labelsize=8, length=0, colors='grey')
cb1.outline.set_edgecolor('grey')
cb1.outline.set_linewidth(0.3)

# Colorbar 2 - Plasma with extend='neither'
ax2 = fig.add_axes([0.1, 0.35, 0.8, 0.08])
cmap2 = plt.cm.plasma
norm2 = mcolors.Normalize(vmin=100, vmax=200)
cb2 = mcolorbar.ColorbarBase(ax2, cmap=cmap2, norm=norm2,
                              orientation='horizontal', extend='neither')
cb2.ax.tick_params(labelsize=8, length=0, colors='grey')
cb2.outline.set_edgecolor('grey')
cb2.outline.set_linewidth(0.3)

# Legend with square patches
ax3 = fig.add_axes([0.1, 0.05, 0.8, 0.18])
ax3.set_xlim(0, 3)
ax3.set_ylim(0, 1)
ax3.axis('off')

colors = ['green', 'sandybrown', 'lightgrey']
labels = ['forest', 'desert', 'tundra']
square_size = 0.35
y_bottom = 0.35

for i, (color, label) in enumerate(zip(colors, labels)):
    x = i + 0.5 - square_size / 2
    rect = plt.Rectangle((x, y_bottom), square_size, square_size,
                          facecolor=color, edgecolor='grey', linewidth=0.3)
    ax3.add_patch(rect)
    ax3.text(x + square_size / 2, y_bottom - 0.15, label,
             ha='center', va='top', fontsize=8, color="grey")

plt.savefig(THUMBNAILS_DIR/"04-styles.png", transparent=True)

In [13]:
fig = ekp.Figure(rows=2, columns=2, figsize=(2, 2), domain=["France", "Spain"])

for i, fmt in enumerate(["GRIB", "netCDF", "xarray", "NumPy"]):
    fig.add_map().contourf(temperature, units="celsius", style="auto").title(fmt, fontsize=10, fontweight="bold", color="grey")

fig.coastlines()

# fig.show()
fig.save(THUMBNAILS_DIR/"02-input-output-formats.png", transparent=True)